In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from new_experiment.best_models.cnn import StandardCNN

print("PyTorch:", torch.__version__)


In [ ]:
SEED = 42

NUM_CLASSES = 200
SELECTED_CLASSES = list(range(NUM_CLASSES))
TRAIN_IMAGES_PER_CLASS = 500
VAL_IMAGES_PER_CLASS = 50

BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 90

MAX_LR = 0.06
SGD_WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.10

FEATURE_DROPOUT = 0.40
BLOCK3_DROPOUT = 0.15
BLOCK4_DROPOUT = 0.25

EARLY_STOPPING_PATIENCE = 30
SAVE_PATH = f"ce_only_{NUM_CLASSES}.pth"
# SAVE_PATH = f"best_models/ce_aug_mix_{NUM_CLASSES}.pth"
# SAVE_PATH = f"best_models/ce_aug_mix_jsd_12_{NUM_CLASSES}.pth"
TINY_ROOT = "../../data/tiny_imagenet"



In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available(): DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available(): DEVICE = torch.device("mps")
else: DEVICE = torch.device("cpu")

print("Device:", DEVICE)


In [ ]:
model = StandardCNN(num_classes=NUM_CLASSES, feature_dropout=FEATURE_DROPOUT, block3_dropout=BLOCK3_DROPOUT, block4_dropout=BLOCK4_DROPOUT).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable parameters (M): {trainable_params / 1e6:.3f}M")

x = torch.randn(2, 3, 64, 64, device=DEVICE)

model.eval()
with torch.no_grad():
    r1 = model.level1(x)
    r2 = model.level2(r1)
    r3 = model.level3(r2)
    r4 = model.level4(r3)
    features = torch.flatten(model.global_pool(r4), 1)
    logits = model(x)

print("Input:", x.shape)
print("r1:", r1.shape)
print("r2:", r2.shape)
print("r3:", r3.shape)
print("r4:", r4.shape)
print("Features:", features.shape)
print("Logits:", logits.shape)

model.train()


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.SGD(model.parameters(), lr=MAX_LR, momentum=0.9, weight_decay=SGD_WEIGHT_DECAY, nesterov=True)

print("Loss: CrossEntropyLoss")
print("Label smoothing:", LABEL_SMOOTHING)
print("Optimizer: SGD + Nesterov")
print("Weight decay:", SGD_WEIGHT_DECAY)
print("OneCycle max LR:", MAX_LR)
print("Initial scheduled LR:", optimizer.param_groups[0]["lr"])
print("Epochs:", EPOCHS)


In [ ]:
checkpoint = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()